In [2]:
import pandas as pd
import numpy as np

# ==============================================================================
# 1. DATA EXTRACTION AND LOADING
# ==============================================================================

# Define the local source file path
path_to_csv = "exoplanets.csv"

# Load the Kepler Object of Interest (KOI) archive directly
df_raw = pd.read_csv(path_to_csv)

# Clean leading and trailing whitespaces from the column headers
df_raw.columns = df_raw.columns.str.strip()

print("==========================================================================")
print("--- RAW DATASET METADATA & SCHEMA (ORIGINAL COLUMNS) ---")
print("==========================================================================")
# Display structural overview, including memory usage and all columns from the source file
df_raw.info()
print("==========================================================================\n")

# ==============================================================================
# 2. FEATURE SELECTION (BEFORE CLEANING)
# ==============================================================================

# Filter the dataset matrix to isolate exclusively the 11 user-selected columns
project_columns = [
    'kepid', 'kepoi_name', 'kepler_name', 'koi_disposition', 'koi_score',
    'koi_period', 'koi_prad', 'koi_teq', 'koi_insol', 'koi_steff', 'koi_srad'
]
df = df_raw[project_columns].copy()

print("==========================================================================")
print(f"INITIAL ISOLATED SUBSET: {df.shape[0]} rows | {df.shape[1]} columns")
print("==========================================================================\n")

# ==============================================================================
# 3. DATA INTEGRITY AUDIT (DATA TYPES AND NAN COUNTS BEFORE CLEANING)
# ==============================================================================

print("--- DATA TYPES AND INITIAL MISSING VALUES (PRE-CLEANING) ---")
# Build a comprehensive technical audit dataframe showing types, NaNs, and missing ratios
null_report = pd.DataFrame({
    'Data Type': df.dtypes,
    'Null Count (NaN)': df.isnull().sum(),
    'Missing Percentage (%)': round((df.isnull().sum() / len(df)) * 100, 2)
})
print(null_report.to_string())
print("\n" + "-"*74 + "\n")

# ==============================================================================
# 4. HYGIENE PHASE: DEDUPLICATION
# ==============================================================================

# Remove duplicate records based on the unique planetary candidate key 'kepoi_name'
detected_duplicates = df.duplicated(subset=['kepoi_name']).sum()
df = df.drop_duplicates(subset=['kepoi_name'], keep='first')
print(f"[Governance] Removed duplicate records by unique key 'kepoi_name': {detected_duplicates}")

initial_rows = df.shape[0]

# ==============================================================================
# 5. FOCALIZED STRICT CLEANING (ROW DELETION ON SPECIFIC COLUMNS SUBSET)
# ==============================================================================

# Define the precise subset of metrics where missing values trigger a row deletion
strict_metrics_subset = ['koi_score', 'koi_teq', 'koi_insol', 'koi_steff', 'koi_srad']

# Drop the entire row only if any NaN is found inside this specific subset of columns
df_cleaned = df.dropna(subset=strict_metrics_subset).copy()
rows_after_strict_metrics = df_cleaned.shape[0]

print("\n==========================================================================")
print("--- GOVERNANCE AUDIT ---")
print("==========================================================================")
print(f"Deduplicated observations before subset filter: {initial_rows}")
print(f"Records remaining after subset filtering:       {rows_after_strict_metrics}")
print(f"Total rows dropped due to missing core metrics: {initial_rows - rows_after_strict_metrics}")
print("==========================================================================\n")

# ==============================================================================
# 6. TEXTUAL VALUES STANDARDIZATION & RADIUS CHECK
# ==============================================================================

# Critical drop for radius: if 'koi_prad' is missing, classification engineering fails
df_cleaned = df_cleaned.dropna(subset=['koi_prad'])

# Handle missing textual records for the remaining rows in 'kepler_name'
# This prevents dropping confirmed vs candidate context required for business storytelling
df_cleaned['kepler_name'] = df_cleaned['kepler_name'].fillna("Unconfirmed Candidate")
print("[Governance] Categorical text labeling and radius validation completed.\n")

# ==============================================================================
# 7. FEATURE ENGINEERING: NASA METRIC SEGMENTATION
# ==============================================================================

def classify_by_radius_nasa(radius):
    """
    Classifies an exoplanet into standard executive categories based on its
    radius relative to Earth (Earth Radii - R_earth).
    """
    if radius < 1.25:
        return "Terrestrial (Earth-size)"
    elif 1.25 <= radius < 2.0:
        return "Super-Earth"
    elif 2.0 <= radius < 6.0:
        return "Neptunian (Ice Giant)"
    else:
        return "Gas Giant (Jupiter-size)"

# Generate the custom categorical feature for the interactive UI dropdown filters
df_cleaned['planet_type'] = df_cleaned['koi_prad'].apply(classify_by_radius_nasa)

# ==============================================================================
# 8. EXPORT OPTIMIZED DATASET
# ==============================================================================

# Save the final clean data frame to a CSV file for target dashboard ingestion
output_filename = "exoplanets_processed.csv"
df_cleaned.to_csv(output_filename, index=False)
print(f"[Storage] Success! Target analysis file saved as: '{output_filename}'\n")

# ==============================================================================
# 9. RE-CALCULATING RESTRUCTURED ADVANCED DASHBOARD STATISTICS
# ==============================================================================

print("="*74)
print("--- ADVANCED METRICS SUMMARY ---")
print("="*74)

# Statistics Block A: Descriptive metrics grouped by planet types
print("\n[A] GROUPED PHYSICAL METRICS (Mean, Median, Standard Deviation):")
grouped_metrics = ['koi_period', 'koi_prad', 'koi_teq', 'koi_insol']
grouped_stats = df_cleaned.groupby('planet_type')[grouped_metrics].agg(['mean', 'median', 'std']).round(2)
print(grouped_stats.to_string())

# Advanced Block B: Pearson Correlation Matrix
print("\n[B] PEARSON CORRELATION COEFFICIENTS FOR COMPLETED FEATURES:")
numerical_features = ['koi_score', 'koi_period', 'koi_prad', 'koi_teq', 'koi_insol', 'koi_steff', 'koi_srad']
correlation_matrix = df_cleaned[numerical_features].corr().round(3)
print(correlation_matrix.to_string())

# Advanced Block C: Integrity Cross-Tabulation
print("\n[C] STRATEGIC DISPOSITION CROSS-TABULATION (Planet Type vs Validation State):")
disposition_cross = pd.crosstab(df_cleaned['planet_type'], df_cleaned['koi_disposition'], margins=True)
print(disposition_cross.to_string())

--- RAW DATASET METADATA & SCHEMA (ORIGINAL COLUMNS) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9564 entries, 0 to 9563
Data columns (total 49 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   kepid              9564 non-null   int64  
 1   kepoi_name         9564 non-null   object 
 2   kepler_name        2359 non-null   object 
 3   koi_disposition    9564 non-null   object 
 4   koi_pdisposition   9564 non-null   object 
 5   koi_score          8054 non-null   float64
 6   koi_fpflag_nt      9564 non-null   int64  
 7   koi_fpflag_ss      9564 non-null   int64  
 8   koi_fpflag_co      9564 non-null   int64  
 9   koi_fpflag_ec      9564 non-null   int64  
 10  koi_period         9564 non-null   float64
 11  koi_period_err1    9110 non-null   float64
 12  koi_period_err2    9110 non-null   float64
 13  koi_time0bk        9564 non-null   float64
 14  koi_time0bk_err1   9110 non-null   float64
 15  koi_time0bk_err